<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/%D0%B3%D0%BE%D1%82%D0%BE%D0%B2%D0%BE_mb51_zayaviteli_excels-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from google.colab import files

import os
import zipfile

In [5]:
df_mb51 = pd.read_excel('/content/MB51_11092026 - база - Copy.XLSX')

In [9]:
print(df_mb51.columns.tolist())

['Склад', 'Обозначение склада', 'Партия', 'Номер материала', 'RU Tanım', 'TR Tanım', 'Общее количество', 'Стоимость общаяя', 'Базовая ЕИ', 'Валюта', 'Заказ на поставку', 'Заявитель', 'Инвентарный номер', 'Группа закупок', 'Дата ввода ZMB51', 'Продолжительность']


Yukarıdaki liste, DataFrame'inizdeki tüm sütun adlarını göstermektedir. Lütfen bu listeyi kontrol edin ve hangi sütunları çıktı dosyalarında kullanmak istediğinizi belirtin. Özellikle `'﻿Завод'` gibi özel karakterli sütun adlarına dikkat edin, bazen farklı okumalar nedeniyle sorunlar yaşanabilir.

### Her bir `Заявитель` için ayrı Excel dosyaları oluşturma

Bu bölümde, `df_mb51` DataFrame'indeki her benzersiz 'Заявитель' (Talep Eden) değeri için ayrı bir Excel dosyası oluşturulacaktır. Her bir dosya, ilgili Заявитель'e ait tüm verileri içerecektir. Ardından, oluşturulan tüm Excel dosyaları bir zip arşivi halinde sıkıştırılarak indirilebilir hale getirilecektir.

In [16]:
# Gerekli sütun sırasını DataFrame'den doğrudan al
columns_order = df_mb51.columns.tolist()

# Benzersiz Заявитель değerlerini al
zayaviteli_list = df_mb51['Заявитель'].dropna().unique()

# Çıktıları kaydetmek için klasör oluştur (varsa geç)
output_dir = "zayaviteli_excels"
os.makedirs(output_dir, exist_ok=True)

print(f"Toplam {len(zayaviteli_list)} Заявитель için Excel dosyaları oluşturulacak.")

# Her Заявитель için döngü
for z_name in zayaviteli_list:
    # Dosya adı için geçersiz karakterleri temizle
    safe_name = "".join(c if c.isalnum() or c in " ._-" else "_" for c in str(z_name)).strip()
    if not safe_name:
        safe_name = "unknown_zayavitel"

    # Filtreleme
    df_filtered = df_mb51[df_mb51['Заявитель'] == z_name].copy()

    # Sütunları belirlenen sıraya göre düzenle (şimdi tüm mevcut sütunlar dahil)
    df_filtered = df_filtered[columns_order]

    # Excel olarak kaydet
    output_path = os.path.join(output_dir, f"MB51_{safe_name}.xlsx")
    df_filtered.to_excel(output_path, index=False, engine='xlsxwriter')

print(f"✅ Tüm dosyalar '{output_dir}/' klasörüne kaydedildi.")

Toplam 51 Заявитель için Excel dosyaları oluşturulacak.
✅ Tüm dosyalar 'zayaviteli_excels/' klasörüne kaydedildi.


### Oluşturulan Excel Dosyalarını Sıkıştırma ve İndirme

Tüm bireysel Excel dosyaları oluşturulduktan sonra, bu dosyalar tek bir `.zip` arşivi haline getirilecek ve indirilmeye hazır hale getirilecektir.

In [17]:
zip_filename = "zayaviteli_excels.zip"

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(output_dir):
        for file in files_in_dir:
            if file.endswith(".xlsx"):
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, output_dir))

print(f"✅ Tüm Заявитель dosyaları '{zip_filename}' olarak sıkıştırıldı.")

✅ Tüm Заявитель dosyaları 'zayaviteli_excels.zip' olarak sıkıştırıldı.


In [18]:
try:
    files.download(zip_filename)
    print(f"'{zip_filename}' indirme işlemi başlatıldı.")
except Exception as e:
    print(f"Dosya indirme hatası: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'zayaviteli_excels.zip' indirme işlemi başlatıldı.
